# Creating a Golden Dataset — Test Data for AI Agent Evaluation

## Definition of Golden Dataset

One of the most important pieces when building an AI agent evaluation system is the **Golden Dataset**.  
The Golden Dataset serves as **ground truth** — the reference answers used to score agent responses.

## Workflow

```
PDFs → convert to .md with pyzerox → text files → load into LangChain → generate test set with LLM
```

### Main steps
1. **PDF parsing**: Use the `pyzerox` library to convert PDFs to Markdown (`.md`)
2. **Text conversion**: Convert Markdown files to plain text (`.txt`) files
3. **Document loading**: Load documents into LangChain for downstream generation


---

**Note**: The error below appears when Poppler is not installed. Install it with Homebrew: `brew install poppler`

ERROR:root:Error converting PDF to images: Unable to get page count. Is poppler installed and in PATH?

Gemini: https://gemini.google.com/share/2aa89628bab4

# Set up environment

- `nest_asyncio` lets you call `asyncio.run()` inside Jupyter’s already-running event loop.

In [1]:
import nest_asyncio  # Jupyter의 기존 이벤트 루프와 asyncio 호환을 위해 필요
from dotenv import load_dotenv  # .env 파일에서 환경변수(API 키 등)를 로드

nest_asyncio.apply()  # 이벤트 루프 패치 적용
load_dotenv()  # .env 파일의 환경변수를 os.environ에 등록

True

## 2. PDF to Markdown (pyzerox)

`pyzerox` is an OCR-based PDF parsing library. It uses the GPT-4.1 model to recognize text and layout in PDFs, then converts them to Markdown.

- **Input**: PDF files in `documents_with_english_titles/`
- **Output**: `.md` files written to `documents_with_english_titles_markdown/`

In [ ]:
from pyzerox import zerox  # OCR 기반 PDF → 마크다운 변환 라이브러리
import os
import asyncio

model = 'gpt-4.1'  # pyzerox가 PDF를 해석할 때 사용할 LLM 모델

async def main():
    """
    documents_with_english_titles 폴더의 모든 PDF를 마크다운으로 변환합니다. 
    """

    # pyzerox 옵션 설정
    kwargs = {}
    custom_system_prompt = None  # 기본 시스템 프롬프트 사용
    select_pages = None  # None이면 모든 페이지를 변환
    output_dir = 'documents_with_english_titles_markdown'  # 변환 결과 저장 디렉토리

    # 폴더 내 모든 PDF 파일을 순회하며 변환
    for file_path in os.listdir("./documents_with_english_titles"):
        file_path = os.path.join("./documents_with_english_titles", file_path)
        # PDF를 마크다운으로 변환 (비동기 함수 - LLM API 호출 포함)
        result = await zerox(
            file_path=file_path,
            model=model,
            output_dir=output_dir,
            custom_system_prompt=custom_system_prompt,
            select_pages=select_pages,
            **kwargs
        )
        print(f"Converted {file_path} to markdown.")
        print(f'result == {result}')

# asyncio.run()으로 비동기 함수 실행 (nest_asyncio 덕분에 Jupyter에서도 동작)
result = asyncio.run(main())

## 3. Markdown files to text files

Copy each `.md` file to `.txt` with the same content so loaders that expect plain text can use the corpus.

In [ ]:
from pathlib import Path

input_dir = Path('./documents_with_english_titles_markdown')  # 마크다운 파일이 있는 디렉토리
output_dir = Path('./documents_with_english_titles_txt')  # 텍스트 파일을 저장할 디렉토리

# 출력 디렉토리가 없으면 생성
output_dir.mkdir(exist_ok=True)

# md files to text files
for md_file in input_dir.glob("*.md"):
    # read md file
    text_content = md_file.read_text(encoding='utf-8')
    
    # change extension to .txt
    txt_file = output_dir / md_file.with_suffix('.txt').name
    txt_file.write_text(text_content, encoding='utf-8')

## Data model

Pydantic models define the structured shape of generated question–answer pairs (ground truth).

In [2]:

from pydantic import BaseModel, Field
from typing import List

class GroundTruth(BaseModel):
    question: str = Field(description = "The question asked by the user.")
    answer: str = Field(description = "The correct answer to the user'squestion.")

class GroundTruthList(BaseModel):
    ground_truths: List[GroundTruth] = Field(description = "A list of ground truth question-answer pairs.")
    

In [3]:
from langchain_groq import ChatGroq

model = ChatGroq(model_name="openai/gpt-oss-120b", temperature=0)
model_with_structured_output = model.with_structured_output(GroundTruthList)

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from pyarrow import system_memory_pool

system_prompt = """You are an AI assistant that generates a list of question-answer pairs from a given FAQ document. 
Make sure to generate the pairs in Korean."""


prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("user", "Generate question-answer pairs from the following FAQ document:\n\n{faq_document}"),
])

In [5]:
from pathlib import Path

faq_md_path = Path('./documents_with_english_titles_markdown/employee_benefits_and_welfare_faq.md')
faq_text = faq_md_path.read_text(encoding='utf-8')

In [6]:
faq_text

'직원 복리후생 및 복지 FAQ\n=====================\n\n1. 연차 유급 휴가 관련\n-----------------\n\nQ1: 연차 휴가는 어떻게 사용하나요?\n- A: 연차 휴가는 본인 승인을 통해 자유롭게 사용할 수 있으며, 사내 HR 시스템에서 신청 가능합니다.\n\nQ2: 연차는 어떻게 계산되나요?\n- A: 근속 연수에 따라 차등 지급되며, 1년 미만은 월 1일, 1~3년은 15일, 10년 이상은 20일입니다.\n\nQ3: 미사용 연차는 어떻게 처리되나요?\n- A: 연차는 해당 연도에 사용해야 하며, 이월은 불가합니다.\n\n2. 출산 및 육아 휴가 관련\n------------------\n\nQ4: 출산 휴가는 언제부터 사용할 수 있나요?\n- A: 출산 예정일 전후로 자유롭게 사용할 수 있으며, 최대 20일 유급으로 제공됩니다.\n\nQ5: 육아 휴가는 누구에게 적용되나요?\n- A: 모든 직원이 신청할 수 있으며, 최대 1년까지 사용할 수 있으며, 첫 3개월은 급여의 50%가 지급됩니다.\n\n3. 경조 휴가 및 지원\n--------------\n\nQ6: 경조 휴가는 어떤 경우에 사용할 수 있나요?\n- A: 본인 결혼, 자녀 출생, 부모/배우자 사망 등 경조사 사유에 따라 휴가가 부여됩니다.\n\nQ7: 경조사 지원금은 어떻게 신청하나요?\n- A: HR 시스템을 통해 신청할 수 있으며, 지원금은 지정된 계좌로 지급됩니다.\n\n4. 교육 지원\n\nQ8: 외부 교육비는 어떻게 신청하나요?\n- A: 외부 교육 프로그램 신청 후 영수증을 첨부하여 HR 시스템에 신청합니다.\n\nQ9: 온라인 교육은 어떤 플랫폼을 사용할 수 있나요?\n- A: 회사 지정 플랫폼 (Inflearn, Udemy)을 자유롭게 이용할 수 있습니다.\n\n\n5. 문화 활동 및 동호회 지원\n\nQ10: 문화의 날 지원금은 어떻게 사용하나요?\n- A: 문화 활동(영화, 공연, 스포츠) 비용으로 사용 가능하며, 월 최대 50,000원 지원

In [7]:
ground_truth_chain = prompt | model_with_structured_output

gt_list = ground_truth_chain.invoke({"faq_document": faq_text})

/Volumes/Samsung_9100Pro_2TB/Projects/eval_agent/.venv/lib/python3.13/site-packages/langsmith/_internal/_addressing.py:104: LangSmithBetaWarning: Addressing runs with `langsmith.address` is in beta and is enabled per workspace. A workspace without it rejects the run, so the trace is lost rather than falling back to a project. The behavior may change without notice.
  _warn_once(


In [8]:
gt_list

GroundTruthList(ground_truths=[GroundTruth(question='연차 휴가는 어떻게 사용하나요?', answer='연차 휴가는 본인 승인을 통해 자유롭게 사용할 수 있으며, 사내 HR 시스템에서 신청 가능합니다.'), GroundTruth(question='연차는 어떻게 계산되나요?', answer='근속 연수에 따라 차등 지급되며, 1년 미만은 월 1일, 1~3년은 15일, 10년 이상은 20일입니다.'), GroundTruth(question='미사용 연차는 어떻게 처리되나요?', answer='연차는 해당 연도에 사용해야 하며, 이월은 불가합니다.'), GroundTruth(question='출산 휴가는 언제부터 사용할 수 있나요?', answer='출산 예정일 전후로 자유롭게 사용할 수 있으며, 최대 20일 유급으로 제공됩니다.'), GroundTruth(question='육아 휴가는 누구에게 적용되나요?', answer='모든 직원이 신청할 수 있으며, 최대 1년까지 사용할 수 있으며, 첫 3개월은 급여의 50%가 지급됩니다.'), GroundTruth(question='경조 휴가는 어떤 경우에 사용할 수 있나요?', answer='본인 결혼, 자녀 출생, 부모/배우자 사망 등 경조사 사유에 따라 휴가가 부여됩니다.'), GroundTruth(question='경조사 지원금은 어떻게 신청하나요?', answer='HR 시스템을 통해 신청할 수 있으며, 지원금은 지정된 계좌로 지급됩니다.'), GroundTruth(question='외부 교육비는 어떻게 신청하나요?', answer='외부 교육 프로그램 신청 후 영수증을 첨부하여 HR 시스템에 신청합니다.'), GroundTruth(question='온라인 교육은 어떤 플랫폼을 사용할 수 있나요?', answer='회사 지정 플랫폼 (Inflearn, Udemy)을 자유롭게 이용할 수 있습니다.'), GroundTruth(question='문화의 날 지원금은 어떻게 사용하나

In [ ]:
import pandas as pd

data_folder_path = Path('./data')

# model_dump(): data model을 json 형식으로 변환 
chain_df = pd.DataFrame([gt.model_dump() for gt in gt_list.ground_truths])
chain_df.to_csv(data_folder_path / 'FAQ_golden_dataset.csv', index=False)

In [12]:
gt_list.ground_truths[0].model_dump()

{'question': '연차 휴가는 어떻게 사용하나요?',
 'answer': '연차 휴가는 본인 승인을 통해 자유롭게 사용할 수 있으며, 사내 HR 시스템에서 신청 가능합니다.'}

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent(
    model = model,
    tools = [],
    response_format = GroundTruthList,
    system_prompt = system_prompt,
    verbose = True
)

gt_from_agent = agent.invoke({"messages": [HumanMessage(content = "Generate question-answer pairs from the following FAQ document:\n\n{faq_document}")]})

In [ ]:
agent_df = pd.DataFrame([gt.model_dump() for gt in gt_from_agent["structured_response"].ground_truths])